# Explicación y prototipo (OE4 y OE5)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")
assert torch.cuda.is_available(), "el modelo de visión y lenguaje necesita GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO = "github.com/Riv2oo4/Trabajo-De-Graduacion.git"
RAMA = "oe4-oe5-y-reproducibilidad"

# el token sale de los secretos de Colab y se pasa por subprocess, que no
# imprime el comando: así no queda escrito en el notebook ni en la salida
token = userdata.get("GITHUB_TOKEN")

# si una corrida anterior dejó el proceso parado dentro de /content/repo,
# borrarlo deja getcwd sin directorio y ningún subproceso vuelve a arrancar
os.chdir("/content")

!rm -rf /content/repo
subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                f"https://x-access-token:{token}@{REPO}", "/content/repo"], check=True)

# el clon trae lo que esté empujado al remoto: si explicacion.py no aparece,
# falta hacer push
assert os.path.exists("/content/repo/analisis/explicacion.py"), \
    f"el clon de {RAMA} no trae explicacion.py: revisar que el clon no haya fallado"
%cd /content/repo/analisis

## Dependencias

In [ ]:
!pip install -q "transformers>=4.49" accelerate gradio
import transformers
print("transformers", transformers.__version__)

## Datos

In [ ]:
import os

DRIVE = "/content/drive/MyDrive/Corpus-tesis"
ZIP_REALES = f"{DRIVE}/corpus_real.zip"
MANIPULADAS = f"{DRIVE}/manipuladas"
RESULTADOS = f"{DRIVE}/resultados"

assert os.path.exists(ZIP_REALES), ZIP_REALES
assert os.path.isdir(MANIPULADAS), MANIPULADAS
assert os.path.isdir(RESULTADOS), RESULTADOS

In [ ]:
# en disco local: leer desde Drive imagen por imagen es lento
!mkdir -p /content/reales && unzip -q -o "{ZIP_REALES}" -d /content/reales

REALES = "/content/reales/real"
CATALOGO = "/content/repo/corpus/catalogo.csv"
print(len(os.listdir(REALES)), "reales")

## Punto de control y umbral

In [ ]:
from inferencia import Detector, umbral_de_resultados
from reproducibilidad import leer_procedencia

TABLA_OE3 = f"{RESULTADOS}/detector_oe1.csv"
PUNTO = f"{RESULTADOS}/puntos_de_control/imagenet_ajustado.pt"

assert os.path.exists(TABLA_OE3), TABLA_OE3
assert os.path.exists(PUNTO), PUNTO

# la tabla debe venir de una corrida con la semilla ya corregida; si no trae
# procedencia, es de antes y hay que repetir entrenamiento_detector.ipynb
procedencia = leer_procedencia(TABLA_OE3)
assert procedencia is not None, "la tabla del OE3 no tiene procedencia: repetir la corrida"
print(procedencia["fecha"], "| commit", procedencia["git"]["commit"],
      "| semilla", procedencia["semilla"])

umbral = umbral_de_resultados(TABLA_OE3)
detector = Detector(PUNTO, umbral=umbral)
print("umbral", umbral, "| canales", detector.canales)

## Robustez a la degradación

In [ ]:
# cuánto cae el detector cuando la fotografía llega reescalada o recomprimida,
# que es lo que le hace cualquier plataforma de difusión. La degradación se
# aplica por igual a los dos miembros de cada pareja
CMD = (
    "python robustez.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --punto-de-control '{PUNTO}'"
    f" --resultados-detector '{TABLA_OE3}'"
    f" --salida '{RESULTADOS}/robustez.csv'"
)
!{CMD}

In [ ]:
import pandas as pd

rob = pd.read_csv(f"{RESULTADOS}/robustez.csv")
rob[rob.grupo.eq("todos")].pivot(
    index="condicion", columns="conjunto",
    values=["auc", "exactitud_balanceada", "localizacion"])

In [ ]:
# la caída respecto de la condición original, que es la cifra a reportar
base = rob[rob.grupo.eq("todos") & rob.condicion.eq("original")].set_index("conjunto")
tabla = rob[rob.grupo.eq("todos")].copy()
tabla["caida_auc"] = tabla.apply(
    lambda f: round(f.auc - base.loc[f.conjunto, "auc"], 4), axis=1)
tabla[["condicion", "conjunto", "lado_menor", "calidad", "auc",
       "auc_ic_bajo", "auc_ic_alto", "caida_auc"]].sort_values(
    ["conjunto", "condicion"])

## Prueba con una imagen

In [ ]:
from PIL import Image

from dataset_parches import cargar_catalogo
from explicacion import Explicador, ModeloQwen

prueba = cargar_catalogo(CATALOGO, "prueba")
fila = prueba[0]
imagen = Image.open(f"{MANIPULADAS}/{fila['archivo']}").convert("RGB")

analisis = detector.analizar(imagen)
print(fila["archivo"], "| nivel", fila["nivel"])
print("puntaje %.4f" % analisis["puntaje"], "| manipulada", analisis["manipulada"])
print("region", analisis["region"], "| caja real", fila["caja"])

In [ ]:
# primera llamada real al modelo: descarga los pesos, tarda varios minutos
explicador = Explicador(ModeloQwen())
explicacion = explicador.explicar(imagen, analisis)
print(explicacion["texto"])
print("\nfuga de veredicto:", explicacion["fuga_de_veredicto"] or "ninguna")

In [ ]:
detector.recorte(imagen, analisis, margen=32)

## Lote del conjunto de prueba

In [ ]:
CMD = (
    "python explicacion.py"
    f" --catalogo {CATALOGO}"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --punto-de-control '{PUNTO}'"
    f" --resultados-detector '{TABLA_OE3}'"
    f" --salida '{RESULTADOS}/explicacion_oe4.csv'"
    " --particion prueba"
)
!{CMD}

## Resumen del OE4

In [ ]:
import pandas as pd

tabla = pd.read_csv(f"{RESULTADOS}/explicacion_oe4.csv")
print(len(tabla), "filas |", tabla.clase.value_counts().to_dict())
tabla.groupby(["nivel", "clase"]).agg(
    n=("archivo", "size"),
    con_texto=("texto", lambda s: s.notna().sum()),
    region_correcta=("region_correcta", "mean"),
    con_fuga=("fuga_de_veredicto", lambda s: s.notna().sum()),
)

In [ ]:
# las descripciones a revisar a mano: la columna revision queda vacía
pd.set_option("display.max_colwidth", 200)
tabla[tabla.clase.eq("manipulada")][["archivo", "nivel", "region_correcta", "texto"]].head(10)

In [ ]:
# copia para revisar en hoja de cálculo
tabla.to_csv(f"{RESULTADOS}/explicacion_oe4_para_revisar.csv", index=False)
print("escrita en Drive")

## Prototipo

In [ ]:
%cd /content/repo/despliegue
import app
from sistema import cargar_sistema

sistema = cargar_sistema(PUNTO, TABLA_OE3)
app.construir(sistema).launch(share=True)